# Cérvix-AI · Entrenamiento en Google Colab

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.

Estructura esperada en tu Drive (`Mi unidad/cervix-ai/`):
- `data/` → `sipakmed.zip`, `cpsmi2025.zip`, `lbc_hussain.zip` (el nombre del zip será el nombre de la carpeta)
- `splits/` → se crea sola (la partición se guarda aquí para hacerla una sola vez)
- `models/` → se crea sola (aquí quedan los modelos entrenados, aunque se cierre Colab)

Los datasets se descomprimen en el disco temporal de Colab (rápido); lo que importa se guarda en Drive.

## 1. Conectar Drive

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/cervix-ai"
for d in ("data", "splits", "models"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
print("Zips encontrados:", os.listdir(f"{DRIVE}/zips"))

## 2. Traer el código desde GitHub e instalar lo que falta
Colab ya trae PyTorch; no lo reinstales (rompería el soporte de GPU).

In [ ]:
%%bash
cd /content
rm -rf cervix-ai
git clone https://github.com/KaiTm4070/cervix-ai.git
cd cervix-ai
pip install -q timm scikit-learn pyyaml
# La carpeta models/ apunta a Drive: lo entrenado se conserva al cerrar Colab
rm -rf models
ln -s /content/drive/MyDrive/cervix-ai/models models
ls

In [ ]:
%cd /content/cervix-ai
import torch
print("GPU disponible:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

## 3. Descomprimir los datasets (disco local de Colab)

In [ ]:
%%bash
mkdir -p /content/data/raw
for z in /content/drive/MyDrive/cervix-ai/zips/*.zip; do
  n=$(basename "$z" .zip)
  [ -d "/content/data/raw/$n" ] || unzip -q "$z" -d "/content/data/raw/$n"
done
ls /content/data/raw

## 4. Preparar `data/processed`
`processed` debe tener **una carpeta por clase**. Aquí va el script de preparación que unifica los datasets
(pendiente de ver la estructura de tus carpetas). Mientras tanto, para probar el flujo con un solo dataset que ya
tenga una carpeta por clase, descomenta y ajusta la ruta:

In [ ]:
# !ln -s /content/data/raw/<dataset>/<carpeta_que_contiene_las_clases> /content/data/processed
# !ls /content/data/processed

## 5. Configuración para Colab
Usa la misma configuración del repo, pero con las rutas de Colab y Drive.

In [ ]:
import yaml

cfg = yaml.safe_load(open("configs/training.yaml", encoding="utf-8"))
cfg["dataset_dir"] = "/content/data/processed"
cfg["splits_file"] = f"{DRIVE}/splits/splits.json"
# cfg["normal_classes"] = ["..."]   # clases normales, para sensibilidad/especificidad binarias
yaml.safe_dump(cfg, open("configs/colab.yaml", "w", encoding="utf-8"), allow_unicode=True)
print(open("configs/colab.yaml", encoding="utf-8").read())

## 6. Partición (una sola vez)
Si ya existe en Drive, se niega a repetirse. Usa `--force` solo si cambiaste de dataset.

In [ ]:
!python -m src.training.split --config configs/colab.yaml

## 7. Entrenar
Guarda el mejor modelo en `Drive/cervix-ai/models/<run_name>/`. Si Colab se desconecta a la mitad, el entrenamiento no se reanuda: hay que repetirlo desde este paso.

In [ ]:
!python -m src.training.train --config configs/colab.yaml

## 8. Evaluar (una sola vez, con el conjunto de prueba)

In [ ]:
!python -m src.evaluation.evaluate --config configs/colab.yaml

## 9. Comparar con otra arquitectura (ViT)
Cambia el modelo y el nombre de la corrida, y repite los pasos 7 y 8 con el nuevo archivo.

In [ ]:
cfg["model_name"] = "vit_small_patch16_224"
cfg["run_name"] = "vit_small_v1"
yaml.safe_dump(cfg, open("configs/colab_vit.yaml", "w", encoding="utf-8"), allow_unicode=True)
# !python -m src.training.train --config configs/colab_vit.yaml
# !python -m src.evaluation.evaluate --config configs/colab_vit.yaml

## 10. Llevarte el modelo al proyecto
En Drive, `cervix-ai/models/<run_name>/` contiene `model.pt`, `package.json`, `metricas.json` y `matriz_confusion.png`.
Descarga la carpeta del mejor modelo y cópiala a `models/vigente/` en tu PC (no se sube a Git).